# OpenAI python SDK - Responses APIs

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

True

In [2]:

client = OpenAI(
    api_key=os.environ.get("MODEL_KEY"),
    base_url=os.environ.get("BASE_URl"),
 
)

In [38]:
from pydantic import BaseModel
class CodeReview(BaseModel):
    summary: str
    issues: list[str]
    severity: str #low , medium, high

In [ ]:
# SYSTEM_PROMPT = """
# You are a helpful AI coding assistant."""

In [40]:
def add(a: int, b: int) -> int:
    """Returns the sum of a and b."""
    return a + b

In [42]:
response1 = client.responses.parse(
    model=os.environ.get("MODEL_NAME"),
    instructions= "Review the given code, response in struchtured format",
    input="def add(a, b): return a + b",
    text_format=CodeReview
)
# response2 = client.responses.create(
#     model=os.environ.get("MODEL_NAME"),
#     previous_response_id=response1.id,
#     input="show me a [practice example of it]"
# )

In [44]:
review = response1.output_parsed

In [46]:
"Output 1" if 5 > 10 else "Output 2"

'Output 2'

In [ ]:
table= [
    ["Summary", review.summary],
    ["Issues","\n".join(review.issues) if review.issues else "No issues found"],
    ["Severity", review.severity]
]

In [ ]:
from tabulate import tabulate

In [ ]:
print(tabulate(table , headers=["Aspect", "Details"], tablefmt="grid"  ))

+----------+----------------------------------------------+
| Aspect   | Details                                      |
+==========+==============================================+
| Summary  | Function definition                          |
+----------+----------------------------------------------+
| Issues   | The function `add` is not clearly documented |
+----------+----------------------------------------------+
| Severity | Minor                                        |
+----------+----------------------------------------------+


In [ ]:
# import pandas as pd

In [ ]:
# data= review.model_dump()

In [ ]:
# df = pd.DataFrame([data])

In [ ]:
df

In [61]:
from ddgs import DDGS
def web_search(query: str) -> str:
    print(f"Searching web for: {query}")
    results = []
    try:
        with DDGS() as ddgs:
            for r in ddgs.text(f"best LLMs for {query} 2024", max_results=5):
                results.append(f"- {r['title']}: {r['body']} ({r['href']})")
    except Exception as e:
        print(f"Search failed: {e}")
        return "No search results available."
    return "\n".join(results) if results else "No results found."


In [62]:
tools = [
    {
        "type": "function",
        "name": "get_current_weather",
        "description": "Get the current weather in a given location",
        "parameters": {
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description": "The city and state, e.g. San Francisco, CA"
                },
                "unit": {
                    "type": "string",
                    "enum": ["celsius", "fahrenheit"]
                }
            },
            "required": ["location", "unit"]
        }
    }
]

In [64]:
search_response = client.responses.create(
    model=os.environ.get("MODEL_NAME"),
    instructions=SYSTEM_PROMPT,
    tools=tools,
    input="What is the latest AI model",
    tool_choice="auto"
)

In [65]:
reasoning_response = client.responses.create(
    model=os.environ.get("MODEL_NAME"),
    instructions=SYSTEM_PROMPT,
    tools=tools,
    reasoning={
        "effort":"high"
    },
    input="What is the latest AI model",
    tool_choice="auto"
)

In [66]:
list(reasoning_response)

[('id', 'resp_880881'),
 ('created_at', 1775547218.0),
 ('error', None),
 ('incomplete_details', None),
 ('instructions', '\nYou are a helpful AI coding assistant.'),
 ('metadata', {}),
 ('model', 'qwen2.5-coder:3b'),
 ('object', 'response'),
 ('output',
  [ResponseOutputMessage(id='msg_135171', content=[ResponseOutputText(annotations=[], text='{"name": "get_current_weather", "arguments": {"location": {"type": "string", "description": "The city and state, e.g. San Francisco, CA"}, "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]}}}', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase=None)]),
 ('parallel_tool_calls', True),
 ('temperature', 1.0),
 ('tool_choice', 'auto'),
 ('tools',
  [FunctionTool(name='get_current_weather', parameters={'properties': {'location': {'description': 'The city and state, e.g. San Francisco, CA', 'type': 'string'}, 'unit': {'enum': ['celsius', 'fahrenheit'], 'type': 'string'}}, 'required': ['location',